# 3D structure generation for modified oligos
This is a quick guide to setting up a linear workflow for generating 3D structures of chemically modified oligonucleotides in any pattern or combination of modifications for both the strands in a duplex.

## Imports
We begin with all top-level requirements and imports, and ensure our software dependencies are setup correctly.

In [ ]:
from maize.core.workflow import Workflow
from maize.utilities.io import Config
from pathlib import Path
import pandas as pd
from maize.utilities.execution import JobResourceConfig
from maize.steps.mai.gromacs_rna.pnab_oligo import GenOligo
from maize.steps.mai.gromacs_rna.file_utils_genoligo import SavePdb

## Workflow

Maize will look for configurations of the specific softwares, in particular the locations of the software packages requested. These configuration should be contained in a TOML file. Note that you will need to set up the required software yourself. 

In [ ]:
flow = Workflow(name="oligo_gen_3D", cleanup_temp=False, level="debug")

flow.config = Config()
flow.config.update(Path("pnab_config.toml"))

Create the node `GenOligo` that takes sequence data in the format shown in example contained inside `maize/steps/mai/gromacs_rna/data` as `sequence_data.csv`. 

In [ ]:
construct = flow.add(GenOligo, name="GenOligo")

## Input

Here we set up path to input data. `data_dir` should contain input file: `sequence_data.csv` lists the sequence and modifications along the sequence for both the guide and passenger strands. It should also contain a config file `input.yaml` that contains necessary parameters to generate a helical duplex. The parameters can be customized based on the user's requirements. For detailed information on the config file, follow the PNAB documentation at https://gt-nucleicacids.github.io/pnab/html/index.html. An `output` folder should be provided wherein the output files get saved.

`sequence_data.csv` file needs entries strictly in the following order: 

**gseq_base**: Sequence of bases in the guide strand <br>
**gseq_ribose**: Sequence of riboses(sugars) in the guide strand <br>
**gseq_backbone**: Sequence of backbones in the guide strand <br>
**guide_term_5**: either 'n' or 'p'. 'n' implies no phosphate in the 5' terminal end, but instead a 5'OH. 'p' implies including phosphate at the 5' end. <br>
**pseq_base**: Sequence of bases in the passenger strand <br>
**pseq_ribose**: Sequence of riboses(sugars) in the passenger strand <br>
**pseq_backbone**: Sequence of backbones in the passenger strand <br>
**pass_term_5**: either 'n' or 'p'. 'n' implies no phosphate in the 5' terminal end, but instead a 5'OH. 'p' implies including phosphate at the 5' end. <br>



Letters indicating bases and modifications in the sequence:

Bases: <br>
A - Adenine <br>
U - Uracil <br>
G - Guanine <br>
C - Cytosine <br>
T - Thymine <br>
M - 5-methyl cytosine <br>

Sugars: <br>
r - 2'hydroxyl ribose (native RNA sugar) <br>
d - 2'deoxy ribose (native DNA sugar) <br>
m - 2'methoxy ribose <br>
e - 2'methoxy ethyl ribose <br>
f - 2'fluoro ribose <br>
g - glycerol <br>
l - locked sugar <br>

Backbone: <br>
p - phosphodiester (native) <br>
r - phosphorothioate (Rp) <br>
s - phosphorothioate (Sp) <br>

In [ ]:
data_dir = Path("../maize/steps/mai/gromacs_rna/data/")
df_seq = pd.read_csv(f"{data_dir}/sequence_data.csv")

#Converting the input sequence dataset to a dictionary. The input should be in this format.
df_dict = {}
for index, row in df_seq.iterrows():
    df_dict[f"mol{index+1}"] = list(row)

#Path to output folder
output = Path("/path/output_pdbs")

Here we set up input and parameters for the node.

In [ ]:
construct.inp_dict.set(df_dict)
construct.inp_data_path.set(data_dir)

## Node to save output structure files

Creating nodes to save output pdb files. `SavePdb` saves all the structure files in one common output folder. We send outputs from the saving node to the output destination folder.

In [ ]:
save_pdb = flow.add(SavePdb[dict[str, Path]], name="save_pdb")
save_pdb.destination.set(output)

Output from `GenOligo` are sent as input to the saving node `SavePdb`.

In [ ]:
flow.connect(construct.out_pdbs, save_pdb.inp)

### Check
If this method doesn't throw an exception, we have connected everything correctly and set all required parameters.

In [ ]:
flow.check() 

### Run
Run the workflow

In [ ]:
flow.execute()